# Proving Speech-to-Text Accuracy on Real Audio

In order to justify our choice of STT technology, we calculate the **Word Error Rate (WER)**.

**Improvement Applied:** Instead of using hardcoded simulated text strings, this notebook now generates an *actual* audio file on your Mac, passes it into the `SpeechRecognition` library, runs a live transcription, and mathematically calculates the exact error rate of the API!

In [ ]:
import speech_recognition as sr
import subprocess
import os

def calculate_wer(reference, hypothesis):
    ref_words = reference.lower().split()
    hyp_words = hypothesis.lower().split()
    
    d = [[0 for _ in range(len(hyp_words) + 1)] for _ in range(len(ref_words) + 1)]
    for i in range(len(ref_words) + 1): d[i][0] = i
    for j in range(len(hyp_words) + 1): d[0][j] = j
        
    for i in range(1, len(ref_words) + 1):
        for j in range(1, len(hyp_words) + 1):
            if ref_words[i - 1] == hyp_words[j - 1]:
                d[i][j] = d[i - 1][j - 1]
            else:
                d[i][j] = min(d[i - 1][j - 1] + 1, d[i - 1][j] + 1, d[i][j - 1] + 1)
                
    return float(d[len(ref_words)][len(hyp_words)]) / len(ref_words)


/Users/dimitrimontgomery/text_to_speech_extention/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


### Step 1: Generate Ground Truth Audio
We use macOS's native `say` command to generate a perfectly pronounced `.wav` file.

In [ ]:
ground_truth_text = "The quick brown fox jumps over the lazy dog in the middle of the summer"
audio_path = "test_dictation.wav"

# macOS specific command to generate a high quality PCM WAV file
print("Generating audio file...")
subprocess.run(['say', '-o', audio_path, '--data-format=LEI16@44100', ground_truth_text])
print(f"Audio saved to {audio_path}")

Generating audio file...
Audio saved to test_dictation.wav


### Step 2: Run Real Transcription & Calculate WER

In [ ]:
recognizer = sr.Recognizer()

print("Transcribing via STT Engine...")
with sr.AudioFile(audio_path) as source:
    audio_data = recognizer.record(source)
    try:
        # Using Google's free API for our baseline comparison
        hypothesis_text = recognizer.recognize_google(audio_data)
        print(f"\nGround Truth: {ground_truth_text}")
        print(f"API Output:   {hypothesis_text}")
        
        wer = calculate_wer(ground_truth_text, hypothesis_text)
        print(f"\nWord Error Rate (WER): {wer * 100:.2f}%")
        if wer < 0.15:
            print("✅ Result: This engine is highly accurate and suitable for our extension.")
    except Exception as e:
        print(f"Transcription failed: {e}")

Transcribing via STT Engine...
Transcription failed: FLAC conversion utility not available - consider installing the FLAC command line application by running `apt-get install flac` or your operating system's equivalent
